In [8]:
from pathlib import Path

test_video_1 = Path("../tests/videos/6.mp4")
test_video_2 = Path("../tests/videos/1.mp4")

In [9]:
# ============================================================
# 使用 seed lite 多模态模型将视频按叙事结构拆解为多个阶段元素
# 利用 instructor 库约束输出为结构化 JSON
# ============================================================
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import instructor
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

from src.video import video_to_base64

load_dotenv(find_dotenv(), override=True)


class ElementContent(BaseModel):
    """单个叙事阶段的内容与时间范围"""

    visual_text: str = Field(
        default="",
        description="该段落画面上的核心叙事文字（不含水印、UI等无关文字）",
    )
    audio_text: str = Field(
        default="",
        description="该段落的音频文本：旁白/台词/对话（纯BGM则返回空字符串）",
    )
    start_time: float = Field(
        default=0,
        description="该阶段在视频中的开始时间（秒）",
    )
    end_time: float = Field(
        default=0,
        description="该阶段在视频中的结束时间（秒）",
    )


class VideoStructure(BaseModel):
    """视频按叙事结构的完整拆解结果"""

    hook: ElementContent | None = Field(
        default=None,
        description="钩子：开头抛出问题/悬念/冲突，目的是抓住观众注意力（通常在前 5-8 秒）",
    )
    setup: ElementContent | None = Field(
        default=None,
        description="铺垫：交代背景、设定情境、介绍前提",
    )
    story: ElementContent | None = Field(
        default=None,
        description="正文：故事主体/事件叙述/观点展开，通常占据视频最大篇幅",
    )
    insight: ElementContent | None = Field(
        default=None,
        description="金句：核心观点/感悟/反转，点睛之笔和传播核心",
    )
    cta: ElementContent | None = Field(
        default=None,
        description="行动号召：引导点赞/关注/转发/评论等互动（关注/订阅等不算，那是噪音）",
    )
    outro: ElementContent | None = Field(
        default=None,
        description="结尾：收束/道别或落版文字",
    )

In [10]:
def extract_transcript(video_path: str | Path) -> VideoStructure:
    """
    使用多模态模型将视频按叙事结构拆解为多个阶段元素。

    参数:
        video_path: 视频文件路径

    返回:
        VideoStructure: 包含 hook/setup/story/insight/cta/outro 各阶段的结构化对象
    """
    client = instructor.from_openai(
        OpenAI(
            api_key=os.getenv("API_KEY"),
            base_url=os.getenv("BASE_URL"),
        )
    )

    video_b64 = video_to_base64(video_path)

    system_prompt = """你是一个专业的短视频内容拆解助手。你分析的对象是"文字叙事类"短视频：
画面主要由动态文字与视觉特效构成，配合背景音乐(BGM)来讲述故事或传达信息，
通常没有旁白/人声对话。

你的任务是将视频按叙事结构拆解为以下 6 个阶段，提取每个阶段的核心叙事文字和时间范围。

————————————————————
文字筛选规则
————————————————————
请严格忽略以下**无关文字**，仅提取创作者意图展示的**核心叙事文字**：
• 水印标记（如 @账号名、频道 logo 旁文字）
• 平台 UI 元素（"订阅""点赞""转发""收藏"等按钮/菜单文字）
• 时间戳、进度条文字
• 角落小字、免责声明、版权声明
• 重复出现且不参与叙事的品牌角标/固定标识
核心叙事文字的特征：占据画面主体、字号较大、有动效、是当前画面的视觉焦点。

————————————————————
6 个叙事阶段说明
————————————————————

1. hook（钩子）
   视频开头 3~8 秒内，抛出问题/悬念/冲突/反常识观点。
   目的是抓住观众注意力，让其产生"然后呢？"的好奇心。
   示例："你知道吗？90%的人都做错了这件事"
         "老板说了一句话，我当场辞职"

2. setup（铺垫）
   交代背景、设定情境、介绍事件前提或人物关系。
   为后续正文展开做铺垫。
   示例："这是我花了三年时间研究出的结果"
         "故事发生在一个偏僻的小镇上"

3. story（正文）
   故事的主体部分：事件经过、观点论证、情节推进、步骤讲解。
   通常占据视频最大的篇幅。
   示例：一系列事件叙述画面 "第一天...第二天..."
        步骤讲解画面 "第一步xxx，第二步xxx..."

4. insight（金句）
   核心观点/感悟/反转/总结，是视频的点睛之笔和传播核心。
   通常出现在高潮处，语气坚定、有力量感。
   示例："人生最大的智慧，就是活在当下"
         "所以，别再为不值得的人浪费时间"

5. cta（行动号召）
   引导用户进行互动操作。
   示例："转发给你关心的人"  "点赞收藏，下次好找"
         "评论区告诉我你的故事" "点击主页了解更多"
   注意：账号名/@标识本身是水印，不属于 cta。

6. outro（结尾）
   收束/道别/落版画面。
   示例："我们下期再见" "谢谢观看" 频道名落版
   注意：仅画面中出现的静态频道名是水印，不属于 outro。
         outro 必须是叙事性的收尾。

————————————————————
输出规则
————————————————————
• 如果某个阶段不存在，该字段返回 null（不要编造）
• audio_text：如果该阶段仅含 BGM 无人声，返回空字符串 ""
• start_time / end_time：估算该阶段在视频中的起止时间（单位：秒）
• 时间范围应连续不重叠，覆盖整个视频
• 忠实还原画面文字，不概括、不改写、不补充"""

    user_prompt = (
        "请分析这个视频，按叙事结构拆解为 hook/setup/story/insight/cta/outro 各阶段。"
        "忽略水印和平台 UI 元素，提取每个阶段的核心叙事文字和音频内容。"
    )

    user_content: list[dict] = [
        {"type": "text", "text": user_prompt},
        {
            "type": "video_url",
            "video_url": {"url": f"data:video/mp4;base64,{video_b64}"},
        },
    ]

    print("🤖 正在调用多模态模型拆解视频结构...")
    response = client.chat.completions.create(
        model=os.getenv("MODEL"),  # type: ignore
        response_model=VideoStructure,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},  # type: ignore
        ],
    )

    return response

In [11]:
# 测试：对 test_video_1 进行结构拆解
result = extract_transcript(test_video_1)

print("=" * 60)
print("📋 视频结构拆解结果")
print("=" * 60)

for field_name in ("hook", "setup", "story", "insight", "cta", "outro"):
    element = getattr(result, field_name)
    status = "✅" if element else "❌ (无此阶段)"
    print(f"\n{status} {field_name.upper()}")
    if element:
        print(f"   ⏱  {element.start_time:.1f}s ~ {element.end_time:.1f}s")
        if element.visual_text:
            print(f"   📺 画面文字: {element.visual_text}")
        if element.audio_text:
            print(f"   🎙 音频文字: {element.audio_text}")
        if not element.visual_text and not element.audio_text:
            print(f"   ⚠️  (内容为空)")

print("\n" + "=" * 60)
print("📋 完整 JSON 输出")
print("=" * 60)
print(result.model_dump_json(indent=2, ensure_ascii=False))

🤖 正在调用多模态模型拆解视频结构...
📋 视频结构拆解结果

✅ HOOK
   ⏱  0.0s ~ 4.0s
   📺 画面文字: 那天我问了个问题，什么东西比10亿更有价值？

✅ SETUP
   ⏱  4.0s ~ 5.5s
   📺 画面文字: 友谊 时光 自由

✅ STORY
   ⏱  5.5s ~ 9.5s
   📺 画面文字: 直到有一天，一个“傻子”回答道，11亿

✅ INSIGHT
   ⏱  9.5s ~ 12.0s
   📺 画面文字: 11亿>10亿

❌ (无此阶段) CTA

✅ OUTRO
   ⏱  12.0s ~ 20.5s
   📺 画面文字: 10亿<11亿，11亿>10亿

📋 完整 JSON 输出
{
  "hook": {
    "visual_text": "那天我问了个问题，什么东西比10亿更有价值？",
    "audio_text": "",
    "start_time": 0.0,
    "end_time": 4.0
  },
  "setup": {
    "visual_text": "友谊 时光 自由",
    "audio_text": "",
    "start_time": 4.0,
    "end_time": 5.5
  },
  "story": {
    "visual_text": "直到有一天，一个“傻子”回答道，11亿",
    "audio_text": "",
    "start_time": 5.5,
    "end_time": 9.5
  },
  "insight": {
    "visual_text": "11亿>10亿",
    "audio_text": "",
    "start_time": 9.5,
    "end_time": 12.0
  },
  "cta": null,
  "outro": {
    "visual_text": "10亿<11亿，11亿>10亿",
    "audio_text": "",
    "start_time": 12.0,
    "end_time": 20.5
  }
}


In [12]:
# ============================================================
# 根据 AI 返回的时间戳，将视频切割为各阶段片段
# 用于人工校验 AI 的时间划分是否准确
# ============================================================
from pathlib import Path

import ffmpeg

_STAGE_ORDER = ["hook", "setup", "story", "insight", "cta", "outro"]


def cut_video_by_structure(
    video_path: str | Path,
    structure: VideoStructure,
    output_dir: str | Path,
) -> dict[str, Path]:
    """
    根据 AI 拆解的时间戳将视频切割为多个片段。
    使用 stream copy（不重新编码），速度极快但切割点可能对齐到最近关键帧。

    参数:
        video_path:   原始视频文件路径
        structure:    AI 返回的 VideoStructure 拆解结果
        output_dir:   输出目录（会自动创建）

    返回:
        dict: {"hook": Path(...), "story": Path(...), ...}
              只包含实际存在的阶段
    """
    video_path = Path(video_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"📁 输出目录: {output_dir.resolve()}")
    clips: dict[str, Path] = {}

    for stage_name in _STAGE_ORDER:
        element: ElementContent | None = getattr(structure, stage_name)
        if element is None:
            continue

        start = max(element.start_time, 0)
        end = element.end_time
        if end <= start:
            print(f"⚠️  {stage_name}: 时间范围无效 ({start:.1f}s ~ {end:.1f}s)，跳过")
            continue

        duration = end - start
        clip_path = output_dir / f"{stage_name}_{start:.1f}s-{end:.1f}s.mp4"

        try:
            ffmpeg.input(str(video_path), ss=start, to=end).output(
                str(clip_path), c="copy"
            ).run(overwrite_output=True, quiet=True)
        except ffmpeg.Error as e:
            print(f"❌ {stage_name}: 切割失败")
            print(e.stderr.decode() if e.stderr else str(e))
            continue

        clips[stage_name] = clip_path
        print(
            f"✅ {stage_name.upper():8s}  {start:5.1f}s ~ {end:5.1f}s"
            f"  ({duration:4.1f}s)  ->  {clip_path.name}"
        )

    print(f"\n🎬 共切割 {len(clips)} 个片段")
    return clips

In [13]:
# 切割视频 — 请确保上面的 result 已经生成
# 将片段输出到 output/clips/ 目录，方便逐一校验
output_clips_dir = Path.cwd() / "output" / "clips"
clips = cut_video_by_structure(test_video_1, result, output_clips_dir)

print("\n" + "=" * 60)
print("📋 各阶段片段与文字对照 — 快速校验")
print("=" * 60)

for stage_name in _STAGE_ORDER:
    element = getattr(result, stage_name)
    clip_path = clips.get(stage_name)
    if element is None:
        print(f"\n❌ {stage_name.upper()}: AI 判定不存在")
    elif clip_path:
        print(f"\n🎬 {stage_name.upper()}  [{clip_path.name}]")
        print(f"   ⏱  {element.start_time:.1f}s ~ {element.end_time:.1f}s")
        if element.visual_text:
            print(f"   📺 {element.visual_text}")
        if element.audio_text:
            print(f"   🎙 {element.audio_text}")

📁 输出目录: D:\HKU\video_structure_transform\backend\notebooks\output\clips
✅ HOOK        0.0s ~   4.0s  ( 4.0s)  ->  hook_0.0s-4.0s.mp4
✅ SETUP       4.0s ~   5.5s  ( 1.5s)  ->  setup_4.0s-5.5s.mp4
✅ STORY       5.5s ~   9.5s  ( 4.0s)  ->  story_5.5s-9.5s.mp4
✅ INSIGHT     9.5s ~  12.0s  ( 2.5s)  ->  insight_9.5s-12.0s.mp4
✅ OUTRO      12.0s ~  20.5s  ( 8.5s)  ->  outro_12.0s-20.5s.mp4

🎬 共切割 5 个片段

📋 各阶段片段与文字对照 — 快速校验

🎬 HOOK  [hook_0.0s-4.0s.mp4]
   ⏱  0.0s ~ 4.0s
   📺 那天我问了个问题，什么东西比10亿更有价值？

🎬 SETUP  [setup_4.0s-5.5s.mp4]
   ⏱  4.0s ~ 5.5s
   📺 友谊 时光 自由

🎬 STORY  [story_5.5s-9.5s.mp4]
   ⏱  5.5s ~ 9.5s
   📺 直到有一天，一个“傻子”回答道，11亿

🎬 INSIGHT  [insight_9.5s-12.0s.mp4]
   ⏱  9.5s ~ 12.0s
   📺 11亿>10亿

❌ CTA: AI 判定不存在

🎬 OUTRO  [outro_12.0s-20.5s.mp4]
   ⏱  12.0s ~ 20.5s
   📺 10亿<11亿，11亿>10亿
